In [1]:
%run -i ../../python_scripts/nb_setup.py

GPU Available: False


#### Choose the dataset you want to use :
* must be data the model did not see at training
* first column : __y_true__ $\in \{0,1\}$ are the samples labels  
* second column : __y_pred__ are the predicted class of samples ($\in \{0,1\}$ too)
* third column : __kappa_f__ is confidence function for each sample, $\in \mathbb{R}$

In [2]:
sgp_df_SR = pickle.load(open("sgp_set_cnn", "rb"))
print(
    "N =",
    sgp_df_SR.shape[0],
    "Proportion of 1s: ",
    np.round(sgp_df_SR.y_true.sum() / sgp_df_SR.shape[0], 2),
)
sgp_df_SR.head(3)

N = 6788 Proportion of 1s:  0.33


,y_true,y_pred,kappa
0,0.0,0.0,0.985617
1,1.0,1.0,0.899486
2,1.0,1.0,0.952112


In [3]:
sgp_df_MCD = pickle.load(open("sgp_set_cnn_MCD", "rb"))
sgp_df_MCD.head(3)

,y_true,y_pred,kappa
0,0.0,0.0,-0.000600
1,1.0,1.0,-0.003494
2,1.0,1.0,-0.001536


In [4]:
num_targets = 50  # number of targets r* to consider when drawing metric/coverage of metric/theta curves
theta_min_SR, theta_max_SR = 0.5, 1  # Sn-independent grid
theta_min_MCD, theta_max_MCD = -0.05, 0  # Sn-independent grid

Plot lines configuration 

In [5]:
lines_list_1_SR, lines_list_2_SR, lines_list_1_MCD, lines_list_2_MCD, lines_list_2 = (
    lines()
)

### 2.1. __FPR__

In [6]:
metric_targets = np.linspace(0.001, 0.2, num=num_targets)

In [ ]:
failures = {"SR": [], "MCD": []}

for s in range(100):

    train_set_SR, test_set_SR = train_test_split(
        sgp_df_SR, seed=s
    )  # drawing 3/4 for bounds fitting, 1/4 for bounds testing
    train_set_MCD, test_set_MCD = train_test_split(
        sgp_df_MCD, seed=s
    )  # same with MCD kappa confidence function
    results_SR = sgp_at_targets(
        train_set_SR,
        test_set_SR,
        metric_targets=metric_targets,
        metric="FPR",
        mode="greedy",
        theta_min=theta_min_SR,
        theta_max=theta_max_SR,
    )
    results_MCD = sgp_at_targets(
        train_set_MCD,
        test_set_MCD,
        metric_targets=metric_targets,
        metric="FPR",
        mode="greedy",
        theta_min=theta_min_MCD,
        theta_max=theta_max_MCD,
    )
    all_results = {"SR": results_SR, "MCD": results_MCD}

    failures["SR"].append(
        all_results["SR"]
        .loc[all_results["SR"].metric_bound < all_results["SR"].test_metric]
        .shape[0]
    )
    failures["MCD"].append(
        all_results["MCD"]
        .loc[all_results["MCD"].metric_bound < all_results["MCD"].test_metric]
        .shape[0]
    )

In [12]:
failure_nb

{'SR': [0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  2,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  4,
  0,
  0,
  0,
  0,
  2,
  0,
  0,
  5,
  2,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0],
 'MCD': [0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  3,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  5,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  3,
  1,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
 

In [ ]:
sum(failure_nb["SR"])

15

In [ ]:
sum(failure_nb["MCD"])

12

In [13]:
pickle.dump(failure_nb, open("failures.pkl", "wb"))